# LangGraph Commercial Assistant — Demo

Demonstrates the **LangGraph** implementation: a low-level `StateGraph` with explicit
nodes, conditional edges, checkpointing, and Human-in-the-Loop (HITL) via `interrupt()`.

## Scenarios
1. List leads
2. Add a new lead
3. Guardrail — out-of-scope question
4. HITL — status update → `lost` (approve flow)
5. HITL — email draft (cancel flow)
6. Conversation memory — same thread across multiple turns

Compare with the Deep Agents equivalent: `deep_agents/demo.ipynb`.

## Setup

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())  # searches parent directories for .env

# ── LangSmith tracing ─────────────────────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "lab-06-langgraph"

# ── Provider / model ──────────────────────────────────────────────────────────
PROVIDER = "anthropic"        # "anthropic" | "openai" | "google"
MODEL    = "claude-sonnet-4-6"  # None → uses provider default

print(f"Provider : {PROVIDER}")
print(f"Model    : {MODEL}")
print(f"Project  : {os.environ['LANGCHAIN_PROJECT']}")

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.types import Command


def show(messages):
    """Print conversation messages in a readable format."""
    for msg in messages:
        role = type(msg).__name__.replace("Message", "")
        content = msg.content
        # Normalize Google Gemini's structured content blocks
        if isinstance(content, list):
            content = " ".join(
                b.get("text", "") for b in content
                if isinstance(b, dict) and b.get("type") == "text"
            )
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[{role:8}] → {tc['name']}({tc['args']})")
        if content:
            preview = content[:500] + ("…" if len(content) > 500 else "")
            print(f"[{role:8}] {preview}")

## Agent Architecture

The LangGraph agent is a `StateGraph` with three nodes:
- **`agent`** — calls the LLM with tool bindings
- **`hitl`** — interrupts execution for human review on sensitive actions
- **`tools`** — executes approved tool calls via `ToolNode`

HITL is triggered by `_find_hitl_call()` which checks:
- `generate_email_draft_tool` (always)
- `update_lead_status_tool` when `new_status='lost'` **(conditional logic)**

Other status transitions (e.g. `prospect → qualified`) execute **without** interruption.
This fine-grained control is a key advantage of the low-level LangGraph approach.

In [ ]:
from langgraph.agent.agent import create_agent

agent = create_agent(provider=PROVIDER, model=MODEL)
print("Agent compiled ✓\n")
agent.get_graph().print_ascii()

## Scenario 1 — List leads

In [ ]:
config_1 = {"configurable": {"thread_id": "demo-lg-list"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Show me all my leads.")]},
    config_1,
)
show(result["messages"])

## Scenario 2 — Add a lead

In [ ]:
config_2 = {"configurable": {"thread_id": "demo-lg-add"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Add a new lead: Jean Dupont from StartupX, "
                    "email j.dupont@startupx.io"
                )
            )
        ]
    },
    config_2,
)
show(result["messages"])

## Scenario 3 — Guardrail

The system prompt restricts the agent to sales pipeline topics.
Out-of-scope questions are refused without calling any tool.

In [ ]:
config_3 = {"configurable": {"thread_id": "demo-lg-guardrail"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="What is the capital of France?")]},
    config_3,
)
show(result["messages"])

## Scenario 4 — HITL: Status update → `lost` (approve)

Marking a lead as `lost` triggers `human_review_node`, which calls `interrupt()` and
pauses the graph. We inspect the interrupt payload then resume with approval.

> **Note:** marking as `qualified` would *not* trigger HITL in this implementation
> (see `_find_hitl_call` in `langgraph/agent/agent.py`). Compare with Deep Agents.

In [ ]:
config_4 = {"configurable": {"thread_id": "demo-lg-hitl-status"}}

# Stream until the graph pauses at the HITL node
list(
    agent.stream(
        {"messages": [HumanMessage(content="Mark lead_002 as lost.")]},
        config_4,
        stream_mode="values",
    )
)

state = agent.get_state(config_4)
print("Paused at node:", state.next)

interrupt_val = state.tasks[0].interrupts[0].value
print("\nPending action:")
print(f"  tool : {interrupt_val['tool_name']}")
print(f"  args : {interrupt_val['tool_args']}")

In [ ]:
# Approve — the tool executes and the agent generates a final response
result = agent.invoke(
    Command(resume={"decision": "approve"}),
    config_4,
)
show(result["messages"][-3:])

## Scenario 5 — HITL: Email draft (cancel)

`generate_email_draft_tool` always triggers HITL — regardless of lead status.
Here we cancel the action and observe the agent's graceful handling.

In [ ]:
config_5 = {"configurable": {"thread_id": "demo-lg-hitl-email"}}

list(
    agent.stream(
        {
            "messages": [
                HumanMessage(content="Generate a follow-up email for lead_003.")
            ]
        },
        config_5,
        stream_mode="values",
    )
)

state = agent.get_state(config_5)
interrupt_val = state.tasks[0].interrupts[0].value
print("Pending action:")
print(f"  tool : {interrupt_val['tool_name']}")
print(f"  args : {interrupt_val['tool_args']}")

In [ ]:
# Cancel — the agent receives the feedback and responds accordingly
result = agent.invoke(
    Command(resume={"decision": "cancel", "feedback": "Wrong lead, please wait."}),
    config_5,
)
show(result["messages"][-3:])

## Scenario 6 — Conversation memory

The `MemorySaver` checkpointer persists the full message history in memory across
invocations on the same `thread_id`. The agent remembers prior turns.

> State is in-memory only — it resets on kernel restart.

In [ ]:
config_6 = {"configurable": {"thread_id": "demo-lg-memory"}}

# Turn 1
result = agent.invoke(
    {"messages": [HumanMessage(content="List all prospect leads.")]},
    config_6,
)
print("=== Turn 1 ===")
show(result["messages"][-2:])

# Turn 2 — agent references the previous response
result = agent.invoke(
    {"messages": [HumanMessage(content="How many leads did you just show me?")]},
    config_6,
)
print("\n=== Turn 2 ===")
show(result["messages"][-2:])

## LangSmith Traces

All runs above were sent to the **`lab-06-langgraph`** project.

Open [LangSmith](https://smith.langchain.com) → select the project to inspect:
- Token usage and cost per run
- Latency breakdown by node
- Tool call sequence
- Full message history

These traces will be referenced in `comparison.ipynb` alongside the Deep Agents traces.

In [ ]:
print(f"LangSmith project : {os.environ['LANGCHAIN_PROJECT']}")
print("URL               : https://smith.langchain.com")